# Linear Regression - Einstieg

Dieses Jupyter Notebook zeigt die Berechnung der Linearen Regression aus unterschiedlichen Python Packages: 
1. mit `scipy.stats.linregress` [scipy.stats.linregress](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html) 
1. mit Seaborn 
1. mit `sklearn.linear_model.LinearRegression` [sklearn.linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) und
1. mit [statsmodels-ols](https://www.statsmodels.org/dev/examples/notebooks/generated/ols.html).

-----
2026-06-15 ug Version 1.5


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy
import sklearn

Version Scipy - Scientific Python

In [ ]:
scipy.__version__

Version Sklear - scikit-learn - Machine Learning in Python

In [ ]:
sklearn.__version__

### Vorbereitung Erzeugen eines künstlichen Datensatzes

Wir erzeugen uns künstliche Daten, von denen wir genau wissen wie sie erzeugt wurden. 
<br>
Wir kennen damit das zugrunde liegende Modell und die "wahren" Parameter dieses Modells. 

$
y_i(x_i) = \beta_0 + \beta_1 x_i + \epsilon_i
$
mit $i=0 .. N-1$

Variablen:
- unabhängige Variable $x_i$  
- abhängige Variable $y_i$

Modell Parameter:
- $\beta_0$ - true_intercept
- $\beta_1$ - true_slope

Störung:
- $\epsilon_i$ - unkorreliert und normalverteilt $\sim \mathcal{N}(\mu,\,\sigma^{2})$ mit Erwartungswert $\mu=0$ (mittelwertfrei) und konstanter Varianz $\sigma^2$

Wir definieren die "wahren" Parameter:

In [ ]:
beta0_true = 2.0    # intercept
beta1_true = 1.6    # slope

#beta1_true = 0    # slope - alternativ

sigma2_true = 0.1  # Varianz der Fehlerterme

Wir erzeugen $N=$ 20 Datenpaare $(x_i,y_i)$ der unabhängigen $x_i$  und der abhängigen Variablen $y_i$:

Später werden wir diesen Wert erhöhen und sehen wie die Schätzung davon beeinflußt wird.

In [ ]:
N = 20

Zufallsgenerator auf einen festen Wert initialisieren - damit wir reproduzierbar die gleichen Zufallszahlen bekommen.

Dies ist wichtig z.B. bei einer Fehlersuche.

In [ ]:
np.random.seed(42)  

Die unabhängige Variable $x_i$ nehmen wir gleichförmig verteilt an.

Gleichförmigverteilte Zufallszahlen können mit [`np.random.rand()`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.rand.html) erzeugt werden:

In [ ]:
x = np.random.rand(N)   # gleichförmig verteilt

Die abhängige Variable $y_i$ (`y_true`) ohne Fehler wird nach $y_i(x_i) = \beta_0 + \beta_1 x_i$ berechnet.

Dies ist der deterministische Anteil der abhängigen Variablen.

In [ ]:
y_true = beta0_true + beta1_true*x

Die Fehler $\epsilon_i$ (`err_true`) werden als normalverteile, unkorrelierte Zufallswerte erzeugt.

Dazu wird die Funktion [`numpy.random.normal(loc,scale)`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.normal.html) mit den Parametern aufgerufen:
- `loc` - Erwartungswert $\mu=0$ (mittelwertfrei)
- `scale` - Standard deviation $\sigma$

Dies ist der stoachstische Anteil der abhängigen Variablen.

In [ ]:
err_true = np.random.normal(0, np.sqrt(sigma2_true), N)

Die abhängige Variable $y$ ist die Summe des deterministischen $y_{\text{true}}$ und stoachtischen Anteils $\text{err}_{\text{true}}$:

$y_i(x_i) = y_{\text{true}} + \text{err}_{\text{true}}= \beta_0 + \beta_1 x_i + \epsilon_i$

In [ ]:
y = y_true + err_true

Wir schauen uns die künstlich erzeugten Daten in einem Scatterplot an:

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x,y, color = 'blue', label='Daten mit Störung')
plt.scatter(x,y_true,color='red', label='Daten ohne Störung')
#plt.plot(x,y_true,color='lime',label='y_true')
plt.grid(True)
plt.title('erzeugte Daten für die lineare Regression')
plt.ylabel('abhängige Variable $y$')
plt.xlabel('unhängige Variable $x$')
plt.legend();

-----
### Statistische Untersuchung des Fehler $\epsilon_i$ 


Der Fehler $\epsilon_i$ soll mittelwertfrei sein.

Wir berechnen den arithmetischen Mittelwert und den Median. Diese sollten nahe bei Null liegen.

In [ ]:
np.mean(err_true)

In [ ]:
np.median(err_true)

Das ist relativ gut erfüllt.

Die Varianz des Fehlers $\epsilon_i$ soll der Varianz $\sigma^2$ entsprechen

In [ ]:
np.var(err_true), sigma2_true

Dies kommt einigermaßen hin.

Die Standartweichung is die Quadratwurzel aus der Varianz; bzw. die Varianz ist das Quadrat der Standardabweichung:
<br>
$ s = \sqrt{\sigma}$ ; $ \sigma = s^2$

In [ ]:
np.std(err_true)

In [ ]:
np.std(err_true)**2

#### Korrelationskoeffizient

[Korrelationskoeffizient-Wikipedia](https://de.wikipedia.org/wiki/Korrelationskoeffizient)

Pearson Korrelationskoeffizient 
[`np.corrcoef()`](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html)
kann Werte zwischen − 1 und + 1 annehmen. 
- Bei einem Wert von + 1 (bzw. -1) besteht ein vollständig positiver (bzw. negativer) linearer Zusammenhang zwischen den betrachteten Merkmalen. 
- Wenn der Korrelationskoeffizient den Wert 0 aufweist, hängen die beiden Merkmale überhaupt nicht linear voneinander ab.



Wie stark ist der Fehler `err_true` mit dem unabhängigem Variablen  $x_i$ korreliert?



In [ ]:
np.corrcoef(err_true,x)

Es liegt eine schwache Korrelation zwischen der unabhängigen Variablen `x` und des Fehlers  `err_true` vor.

### Autocorrelation des Fehlers

[Autokorrelation-Wikipedia](https://de.wikipedia.org/wiki/Autokorrelation)

Wir plotten die Autokorrelation-Funktion 
[matplotlib.pyplot.acorr()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.acorr.html)
des Fehlers `err_true`:

In [ ]:
plt.acorr(err_true, maxlags = 10);

Wir sehen bei 0.0 einen großen Wert, der 1.0 ist.
<br>
Die muss so sein, da das Signal mit sich selber korreliert ist.
<br>
Die Werte daneben geben wie das Signal mit einem verzögerten Signal verzögert ist.

----
### Lineare Regression mit `scipy.stats.linregress`

Linear least-squares Regression:
[`scipy.stats.linregress()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html) 
  

In [ ]:
res = scipy.stats.linregress(x, y)
res

In [ ]:
print(f"slope:     (beta_1) {res.slope:6.3f} (stderr:{res.stderr:3.3f}) wahrer Parameter:{beta1_true:3.3f}")
print(f"intercept: (beta_0) {res.intercept:6.3f} (stderr:{res.intercept_stderr:3.3f}) wahrer Parameter:{beta0_true:3.3f}")
print(f"rvalue: {res.rvalue} (Correlation coefficient)")
print(f"pvalue: {res.pvalue} (Two-sided p-value for a hypothesis test whose null hypothesis is that the slope is zero)")

geschätzte Parameter:
- Steigung (slope)
  - Der Schätzwert der Steigung $\hat{\beta_1}$ beträgt 1.437; der wahre Wert $\beta_1$ beträgt 1.6.
  - Der Schätzwert der Steigung $\hat{\beta_1}$ hat eine Standardabweichung von 0.219. 
- Achsabschnitt (intercept)
  - Der Schätzwert der Steigung $\hat{\beta_0}$ beträgt 1.988; der wahre Wert $\beta_0$ beträgt 2.000.
  - Der Schätzwert der Steigung $\hat{\beta_0}$ hat eine Standardabweichung von 0.120. 
  
Diese Standardabweichung der geschätzten Parameter werden wir weiter unten für die Berechnung der Konfidenzintervalle nutzen.

Den Korrelationskoeffizienten können wir zur Berechung des Bestimmungsmaßes $R^2$ heranziehen.

Mit dem p-Wert [p-Wert - Wikipedia](https://de.wikipedia.org/wiki/P-Wert) wird überprüft, ob die geschätzte Steigung $\hat{\beta_1}$ **„statistisch signifikant“** ist.

Die geschätzte Steigung $\hat{\beta_1}$ ist **„statistisch signifikant“**, wenn der p-Wert kleiner als einem festlegten Signifikanzniveaus z.B. 5% -> 0.05 ist.


In [ ]:
res.pvalue < 0.05

Die geschätze abhängige Variable $\hat{y_i}$ (auch Prediktor) wird berechnet aus $\hat{y_i}(x_i) = \hat{\beta_0} + \hat{\beta_1} x_i$

In [ ]:
y_hat = res.intercept + res.slope*x

Wir plotten die geschätze Geraden im Scatterplot:

In [ ]:
plt.figure(figsize=(10,6))

# senkrechte Linien
for i in range(len(x)):
    lineXdata = (x[i], x[i]) # same X
    lineYdata = (y[i], y_hat[i]) # different Y
    plt.plot(lineXdata, lineYdata,color='grey')

# unabhängig und abhängige Variable
plt.scatter(x,y)

# Gerade mit wahren Parametern
plt.plot(x,y_true,label='y_true',color='lime')

# Gerade mit geschätzen Parametern
plt.plot(x,y_hat,label='y_hat',color='red')

plt.grid(True)
plt.title('linear Regression ')
plt.ylabel('abhängige Variable $y$')
plt.xlabel('unhängige Variable $x$')
plt.legend()


----
### Residuen - Fehler zwischen abhängiger und geschätzer abhängiger Variablen

Resiuden $ \hat{\epsilon_i} = y_i - \hat{y}_i $


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(x,err_true,label=r'err_true $\epsilon_i$',color='lime')
plt.scatter(x,y-y_hat,label=r'Residuen $\hat{\epsilon_i}=y_i-\hat{y_i}$',color='red')
plt.grid(True)
plt.title('')
plt.ylabel('Fehler, Residuen')
plt.xlabel('unhängige Variable $x$')
plt.legend();

### Metriken für die Residuen $ \hat{\epsilon_i} = y_i - \hat{y}_i $

-----
#### Mean Absolute Error
$
\text{MAE} = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|
$


In [ ]:
residuum = y-y_hat
MAE = np.sum(np.abs(residuum))/len(residuum)
MAE

[`sklearn.metrics.mean_absolute_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)

In [ ]:
MAE = sklearn.metrics.mean_absolute_error(y,y_hat)
MAE

----
#### Mean Squared Error
$
\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$

In [ ]:
residuum = y-y_hat
MSE = np.sum(residuum**2)/len(residuum)
MSE

dies entspricht der Varianz [`np.var()`](https://numpy.org/doc/stable/reference/generated/numpy.var.html) der Residuen:

In [ ]:
np.var(y-y_hat)

[`sklearn.metrics.mean_squared_error()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)

In [ ]:
MSE = sklearn.metrics.mean_squared_error(y,y_hat)
MSE

#### Root Mean Squared Error

$
\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}
$

In [ ]:
np.sqrt(sklearn.metrics.mean_squared_error(y,y_hat))

Die Residuen sind mittelwertfrei:

In [ ]:
np.mean(y-y_hat)

----
### Bestimmheitsmass/Determinationskoeffizient $R^2$

Kennzahl zur Beurteilung der Anpassungsgüte einer Regression 
[Bestimmungsmass-Wikipedia](https://de.wikipedia.org/wiki/Bestimmtheitsma%C3%9F)

Das Bestimmungsmaß $R^2$ gibt an, wie viel Streuung in den Daten durch ein vorliegendes lineares Regressionsmodell „erklärt“ werden kann.

$
R^2 = \frac{\text{ESS}}{\text{TSS}} = 1 - \frac{\text{RSS}}{\text{TSS}}
$


mit 
- total sum of squares (TSS) - auch SQT [Totale Quadratsumme-Wikipedia](https://de.wikipedia.org/wiki/Totale_Quadratsumme)
    
    $
    \text{TSS} = \sum_{i=1}^{N} (y_i - \bar{y})^2  
    $
    mit Mittelwert der abhängigen Variablen
    $
    \bar{y} = \sum_{i=1}^{N} (y_i)  
    $
    
-  explained sum of squares (ESS) - auch SQE [Erklärte Quadratsumme-Wikipedia](https://de.wikipedia.org/wiki/Erkl%C3%A4rte_Quadratsumme)

    $
    \text{ESS} = \sum_{i=1}^{N} (\hat{y}_i - \bar{y})^2 = \sum_{i=1}^{N} (\hat{y}_i - \bar{\hat{y}})^2
    $


- residual sum of squares (RSS) - SQR [Residuenquadratsumme-Wikipedia](https://de.wikipedia.org/wiki/Residuenquadratsumme)
    
    $
    \text{RSS} = \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
    $
----
Interpretation:

Das Bestimmtheitsmaß lässt sich mit 100 % multiplizieren, um es in Prozent anzugeben: 100 % 
<br>
$R^2$ ist dann der prozentuale Anteil der Streuung in $y$ der durch das lineare Modell „erklärt“ wird, und liegt daher zwischen:
-   0 % (oder 0): schlechtes Modell und
- 100 % (oder 1): perfektes Modell.

<!---
residual standard error (RSE)

$
RSE = \sqrt{\frac{1}{N-2}RSS} =  \sqrt{\frac{1}{N-2}  \sum_{i=1}^{N} (y_i - \hat{y}_i)^2}
$
-->


Berechnung des Mittelwertes der abhängigen Variablen
    $
    \bar{y} = \sum_{i=1}^{N} (y_i)  
    $

In [ ]:
y_bar = np.sum(y)/len(y)
y_bar

alternative Berechnung mit `np.mean()`

In [ ]:
y_bar = np.mean(y)
y_bar

Berechnung des Mittelwertes der geschätzten abhängigen Variablen
    $
    \bar{\hat{y}} = \sum_{i=1}^{N} (\hat{y_i})  
    $

In [ ]:
y_hat_bar = np.mean(y_hat)
y_hat_bar

Total sum of squares (TSS) - auch SQT [Totale Quadratsumme-Wikipedia](https://de.wikipedia.org/wiki/Totale_Quadratsumme)
    
$
\text{TSS} = \sum_{i=1}^{N} (y_i - \bar{y})^2  
$
mit Mittelwert der abhängigen Variablen
$
\bar{y} = \sum_{i=1}^{N} (y_i)  
$

In [ ]:
TSS = np.sum((y-y_bar)**2)
TSS

 explained sum of squares (ESS) - auch SQE [Erklärte Quadratsumme-Wikipedia](https://de.wikipedia.org/wiki/Erkl%C3%A4rte_Quadratsumme)

$
\text{ESS} = \sum_{i=1}^{N} (\hat{y}_i - \bar{y})^2 = \sum_{i=1}^{N} (\hat{y}_i - \bar{\hat{y}})^2
$


In [ ]:
ESS = np.sum((y_hat-y_hat_bar)**2)
ESS

residual sum of squares (RSS) - SQR [Residuenquadratsumme-Wikipedia](https://de.wikipedia.org/wiki/Residuenquadratsumme)
    
$
\text{RSS} = \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$


In [ ]:
RSS = np.sum((y-y_hat)**2)
RSS

$
R^2 = \frac{\text{ESS}}{\text{TSS}} 
$

In [ ]:
R_squared = ESS/TSS
R_squared

$
R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}
$

In [ ]:
R_squared = 1 -(RSS)/TSS
R_squared

In [ ]:
R_squared = 1 - np.var(y-y_hat)/np.var(y)
R_squared

In [ ]:
print(f"R-squared: {res.rvalue**2}")

[`sklearn.metrics.explained_variance_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.explained_variance_score.html)

In [ ]:
R_squared = sklearn.metrics.explained_variance_score(y,y_hat)
R_squared

[`sklearn.metrics.r2_score()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html)

In [ ]:
R_squared = sklearn.metrics.r2_score(y,y_hat)
R_squared

#### Darstellung von TSS, ESS und RSS

In [ ]:
fig, (ax1,ax2,ax3) = plt.subplots(ncols=3,figsize=(14,6), sharex=True,sharey=True)


fig.suptitle(f'Bestimmungsmass $R^2$={sklearn.metrics.r2_score(y,y_hat):.3f}', fontsize=16)
y_bar = np.mean(y)

# ------------------------------
# total sum of squares (TSS)
for i in range(len(x)):
    lineXdata = (x[i], x[i]) # same X
    lineYdata = (y[i], y_bar) # different Y
    ax1.plot(lineXdata, lineYdata,color='grey')
ax1.scatter(x,y,color='blue',label='$y$')
ax1.axhline(y_bar,label=r'$\bar{y}$',color='cyan')
ax1.scatter(x,y_bar*np.ones_like(x),color='cyan')
ax1.grid(True)
ax1.set_ylabel('abhängige Variable $y$')
ax1.set_xlabel('abhängige Variable $x$')
ax1.legend()
ax1.set_title(f'total sum of squares (TSS)={np.sum((y-y_bar)**2):.3f}') 
    
# --------------------------------------
# explained sum of squares (ESS)
for i in range(len(x)):
    lineXdata = (x[i], x[i]) # same X
    lineYdata = (y_bar, y_hat[i]) # different Y
    ax2.plot(lineXdata, lineYdata,color='grey')
ax2.scatter(x,y,color='blue',label='$y$')
ax2.axhline(y_bar,label=r'$\bar{y}$',color='cyan')
#ax2.plot(x,y_true,label='y_true',color='lime')
ax2.plot(x,y_hat,label='y_hat',color='red')
ax2.scatter(x,y_hat,color='red')
ax2.scatter(x,y_bar*np.ones_like(x),color='cyan')

ax2.grid(True)
#ax2.set_ylabel('abhängige Variable $y$')
ax2.set_xlabel('abhängige Variable $x$')
ax2.legend()
ax2.set_title(f'explained sum of squares (ESS)={np.sum((y_hat-y_hat_bar)**2):.3f}') 

# --------------------------------------
# residual sum of squares (RSS)
for i in range(len(x)):
    lineXdata = (x[i], x[i]) # same X
    lineYdata = (y[i], y_hat[i]) # different Y
    ax3.plot(lineXdata, lineYdata,color='grey')
ax3.scatter(x,y,color='blue',label='$y$')
ax3.axhline(y_bar,label=r'$\bar{y}$',color='cyan')
#ax3.plot(x,y_true,label='y_true',color='lime')
ax3.plot(x,y_hat,label='y_hat',color='red')
ax3.scatter(x,y_hat,color='red')


ax3.grid(True)
#ax3.set_ylabel('abhängige Variable $y$')
ax3.set_xlabel('abhängige Variable $x$')
ax3.legend()
ax3.set_title(f'residual sum of squares (RSS)={np.sum((y-y_hat)**2):.3f}');

fig.tight_layout()

    


---
### Konfidenzintervall/Vertrauensintervalle für geschätze Parameter:

[Konfidenzintervall-Wikipedia](https://de.wikipedia.org/wiki/Konfidenzintervall)

[Studentsche_t-Verteilung-Wikipedia](https://de.wikipedia.org/wiki/Studentsche_t-Verteilung)

[scipy.stats.t](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html)

- pdf(x, df, loc=0, scale=1) - Probability density function.
- cdf(x, df, loc=0, scale=1) - Cumulative distribution function.
- ppf(q, df, loc=0, scale=1) - Percent point function (inverse of cdf — percentiles).

In [ ]:
from scipy.stats import t

In [ ]:
df = 2
print(t.ppf(0.01, df))
x_tmp = np.linspace(t.ppf(0.01, df), t.ppf(0.99, df), 100)
fig,(ax1,ax2) = plt.subplots(ncols=2,figsize=(12,6))

ax1.plot(x_tmp, t.pdf(x_tmp, df),'r-', label='t pdf')
ax1.plot(x_tmp, t.cdf(x_tmp, df),'b-', label='t cdf')
ax1.grid(True)
ax1.legend()

x2_tmp = np.linspace(0, 1, 100)
ax2.plot(x2_tmp, t.ppf(x2_tmp, df),'b-', label='t ppf')
ax2.grid(True)
ax2.legend()
plt.show()

In [ ]:
# Two-sided inverse Students t-distribution

# p - probability, df - degrees of freedom
tinv = lambda p, df: abs(t.ppf(p/2, df))

ts = tinv(0.05, len(x)-2)
print(f"ts: {ts}")

print(f"slope (95%): {res.slope:.6f} +/- {ts*res.stderr:.6f}")
print(f"intercept (95%): {res.intercept:.6f} +/- {ts*res.intercept_stderr:.6f}")

----
### Lineare Regression mit Seaborn


In [ ]:
import pandas as pd
import seaborn as sns

Wir erstellen uns einen Pandas DataFrame aus der unabhängigen $x_i$  und der abhängigen Variablen $y_i$:

In [ ]:
df = pd.DataFrame(data={'x':x,'y':y})

#### Seaborn Linear Regression-Plots

[`seaborn.regplot()`](https://seaborn.pydata.org/generated/seaborn.regplot.html)

[`seaborn.lmplot()`](https://seaborn.pydata.org/generated/seaborn.lmplot.html)

In [ ]:
sns.regplot(data=df,x='x',y='y')

In [ ]:
sns.lmplot(data=df,x='x',y='y')

Es ist nicht möglich die geschätzen Parameter zu bekommen:

https://stackoverflow.com/questions/22852244/how-to-get-the-numerical-fitting-results-when-plotting-a-regression-in-seaborn



Frage: Wie bekomme ich bei Seaborn `sns.regplot(data=df,x='x',y='y')` die geschätzen Parameter?
<br>
Antort: [Perplexity](https://www.perplexity.ai/search/wie-bekomme-ich-bei-seaborn-sn-..Bzn8ccTBiL.7lZZCrLqQ#0)
<br>
Um die geschätzten Parameter (Steigung und Achse) der Regressionsgeraden aus einem Seaborn-Plot mit `sns.regplot(data=df, x='x', y='y')` zu erhalten, gibt es in Seaborn selbst keine eingebaute Funktion. 
<br>
Du kannst jedoch einfach mit `scipy.stats.linregress` die Parameter berechnen. 
<br>
Hier ist ein Beispiel, wie das geht:


In [ ]:
# Parameter berechnen
import scipy
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(df['x'], df['y'])

print("Steigung:", slope)
print("Achsenabschnitt:", intercept)

----
### Lineare Regression mit `sklearn`

[`sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)


In [ ]:
from sklearn import linear_model
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
print(x.shape)
x

unabhängige und abhängige Variable von ein-dimensionalen (20) nach "zwei-dimensional" (20x1) wandeln

[`np.reshape()`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html)

`np.reshape(newshape)`

newshape int or tuple of ints

The new shape should be compatible with the original shape. If an integer, then the result will be a 1-D array of that length. One shape dimension can be -1. In this case, the value is inferred from the length of the array and remaining dimensions.


In [ ]:
X_sklearn = x.reshape((-1,1))
y_sklearn = y.reshape((-1,1))
print(X_sklearn.shape)
X_sklearn

linear regression Object erzeugen

[`sklearn.linear_model.LinearRegression()`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)


In [ ]:
regr = linear_model.LinearRegression()
regr

Modell an die Daten anpassen

In [ ]:
regr.fit(X_sklearn, y_sklearn)

Ergebnisse der Schätzung anzeigen

In [ ]:
print(f"slope: {regr.coef_[0][0]:.3f} beta1_true:{beta1_true:3.3f}")
print(f"intercept: {regr.intercept_[0]:.3f} beta0_true:{beta0_true:3.3f}")
print(f"R_squared: {regr.score(X_sklearn, y_sklearn):.3f}")

Prädiktion - verwenden der geschätzten Parameter

In [ ]:
y_pred = regr.predict(X_sklearn)
y_pred

In [ ]:
regr.predict(np.array(0.5).reshape(-1,1))

In [ ]:
print(f"Mean squared error: {mean_squared_error(y_sklearn,y_pred):.3f}")
print(f"R_squared: {r2_score(y_sklearn,y_pred):.3f}")

In [ ]:
plt.figure(figsize=(10,6))

# senkrechte Linien
for i in range(len(x)):
    lineXdata = (x[i], x[i]) # same X
    lineYdata = (y[i], y_pred[i][0]) # different Y
    plt.plot(lineXdata, lineYdata,color='grey')
plt.scatter(x,y)
plt.plot(x,y_true,label='y_true',color='lime')
plt.plot(x,y_pred,label='y_hat',color='red')
plt.grid(True)
plt.title('linear Regression ')
plt.ylabel('abhängige Variable $y$')
plt.xlabel('abhängige Variable $x$')
plt.legend()

----
### Lineare Regression mit `statsmodels`

Python package _statsmodels_ muss installiert sein.

[statsmodels-ols](https://www.statsmodels.org/dev/examples/notebooks/generated/ols.html)

In [ ]:
import statsmodels.api as sm

In [ ]:
print(x.shape)
x

-> x liegt als Vektor vor.
Für die Schätzung eines Achsenabschnitts wird ein konstanter Vektor mit Einsen angehängt

[`statsmodels.add_constant()`](https://www.statsmodels.org/stable/generated/statsmodels.tools.tools.add_constant.html)

In [ ]:
x_fit = sm.add_constant(x)
print(x_fit.shape)
x_fit

Linear Regression (Ordinary Least Squares) berechnen

[`statsmodels.OLS()`](https://www.statsmodels.org/devel/generated/statsmodels.regression.linear_model.OLS.html)


In [ ]:
fit_results = sm.OLS(y, x_fit).fit()
print(fit_results.summary())

In [ ]:
#dir(fit_results)

Voraussage mit dem Modell berechnen

In [ ]:
Anzahl_Punkte = 100
x_vector = np.linspace(np.min(x), np.max(x), Anzahl_Punkte)
x_vector[:10]

Konstante 1 voranstellen

In [ ]:
eval_x = sm.add_constant(x_vector)
eval_x[:10,:]

Prädiktion Objekt erstellen

In [ ]:
pred_obj = fit_results.get_prediction(eval_x)
pred_obj 

Methoden und Attribute der Prädiktion
[statsmodels - PredictionResult](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.PredictionResults.html)

- `.predicted_mean()` - Mittelwert der abhängigen Variablen
- `.se_mean()` - Mittelwert des Standard Fehler (se = standard error) [Standardfehler-Wiki](https://en.wikipedia.org/wiki/Standard_error)

In [ ]:
# Datenpunkte
plt.scatter(x,y)
# Gerade
plt.plot(eval_x[:,1], pred_obj.predicted_mean,color='red')
# Konfidenzinterval
n_std = 2
plt.fill_between(
        eval_x[:, 1],
        pred_obj.predicted_mean - n_std * pred_obj.se_mean,
        pred_obj.predicted_mean + n_std * pred_obj.se_mean,
        alpha=0.5)
plt.plot(x,y_true,label='y_true',color='lime')
